In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import xgboost as xgb
import joblib
import matplotlib.pyplot as plt

In [2]:
# Define diseases
diseases = ["Flu", "COVID-19", "Pneumonia", "Diabetes", "Heart Disease"]

# Define symptoms
symptoms = ["fever", "cough", "fatigue", "chest_pain", "shortness_of_breath"]

# Create 100 fake patients
np.random.seed(42)  # Same random numbers every time
data = []

for i in range(100):
    # Pick a random disease
    disease = np.random.choice(diseases)
    
    # Create symptom row (all 0s first)
    row = {s: 0 for s in symptoms}
    
    # Add disease-specific symptoms
    if disease == "Flu":
        row["fever"] = 1
        row["cough"] = 1
        row["fatigue"] = 1
    elif disease == "COVID-19":
        row["fever"] = 1
        row["cough"] = 1
        row["fatigue"] = 1
    elif disease == "Pneumonia":
        row["fever"] = 1
        row["cough"] = 1
        row["chest_pain"] = 1
        row["shortness_of_breath"] = 1
    elif disease == "Diabetes":
        row["fatigue"] = 1
    elif disease == "Heart Disease":
        row["chest_pain"] = 1
        row["shortness_of_breath"] = 1
    
    # Add some random noise (10% chance of extra symptom)
    for s in symptoms:
        if np.random.random() < 0.1:
            row[s] = 1
    
    row["disease"] = disease
    data.append(row)

# Convert to DataFrame
df = pd.DataFrame(data)
print(df.head(10))  # Show first 10 rows
print(f"\nDataset shape: {df.shape}")
print(f"\nDisease counts:\n{df['disease'].value_counts()}")

   fever  cough  fatigue  chest_pain  shortness_of_breath        disease
0      0      0        1           0                    0       Diabetes
1      1      1        0           1                    1      Pneumonia
2      0      1        1           0                    0       Diabetes
3      0      0        0           1                    1  Heart Disease
4      1      1        0           1                    1      Pneumonia
5      1      1        1           0                    0            Flu
6      0      0        0           1                    1  Heart Disease
7      1      1        0           1                    1      Pneumonia
8      1      1        1           0                    0       COVID-19
9      0      0        1           0                    1       Diabetes

Dataset shape: (100, 6)

Disease counts:
disease
Diabetes         21
Flu              21
Pneumonia        20
COVID-19         20
Heart Disease    18
Name: count, dtype: int64


In [3]:
# X = features (symptoms), y = target (disease)
X = df.drop("disease", axis=1)  # All columns except disease
y = df["disease"]                # Only disease column

print("Features (X):")
print(X.head())
print(f"\nTarget (y):")
print(y.head())

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% for testing
    random_state=42,    # Same split every time
    stratify=y          # Keep same disease ratios
)

print(f"\nTrain size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

Features (X):
   fever  cough  fatigue  chest_pain  shortness_of_breath
0      0      0        1           0                    0
1      1      1        0           1                    1
2      0      1        1           0                    0
3      0      0        0           1                    1
4      1      1        0           1                    1

Target (y):
0         Diabetes
1        Pneumonia
2         Diabetes
3    Heart Disease
4        Pneumonia
Name: disease, dtype: object

Train size: 80
Test size: 20


In [6]:
# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,    # 100 trees
    max_depth=5,         # Max tree depth
    random_state=42
)

# Train (fit) the model
print("Training Random Forest...")
rf_model.fit(X_train, y_train)
print("Training complete!")

# Predict on test data
rf_predictions = rf_model.predict(X_test)

# Calculate accuracy
rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_f1 = f1_score(y_test, rf_predictions, average='weighted')

print(f"\nRandom Forest Results:")
print(f"  Accuracy: {rf_accuracy:.4f} ({rf_accuracy*100:.1f}%)")
print(f"  F1 Score: {rf_f1:.4f}")

Training Random Forest...
Training complete!

Random Forest Results:
  Accuracy: 0.8000 (80.0%)
  F1 Score: 0.7747


In [5]:
# Create XGBoost model
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

# Train
print("Training XGBoost...")
xgb_model.fit(X_train, y_train)
print("Training complete!")

# Predict
xgb_predictions = xgb_model.predict(X_test)

# Evaluate
xgb_accuracy = accuracy_score(y_test, xgb_predictions)
xgb_f1 = f1_score(y_test, xgb_predictions, average='weighted')

print(f"\nXGBoost Results:")
print(f"  Accuracy: {xgb_accuracy:.4f} ({xgb_accuracy*100:.1f}%)")
print(f"  F1 Score: {xgb_f1:.4f}")


Training XGBoost...


ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2 3 4], got [np.str_('COVID-19') np.str_('Diabetes') np.str_('Flu')
 np.str_('Heart Disease') np.str_('Pneumonia')]

In [7]:
# Method 1: Manual download from Kaggle website
# Method 2: Using opendatasets library
!pip install opendatasets
import opendatasets as od
od.download("https://www.kaggle.com/paultimothymooney/chest-xray-pneumonia")


   ----------- ---------------------------- 2/7 [mdit-py-plugins]
   ----------- ---------------------------- 2/7 [mdit-py-plugins]
   ----------------- ---------------------- 3/7 [kagglesdk]
   ----------------- ---------------------- 3/7 [kagglesdk]
   ----------------- ---------------------- 3/7 [kagglesdk]
   ----------------- ---------------------- 3/7 [kagglesdk]
   ----------------- ---------------------- 3/7 [kagglesdk]
   ----------------- ---------------------- 3/7 [kagglesdk]
   ----------------- ---------------------- 3/7 [kagglesdk]
   ---------------------- ----------------- 4/7 [jupytext]
   ---------------------- ----------------- 4/7 [jupytext]
   ---------------------------- ----------- 5/7 [kaggle]
   ---------------------------------- ----- 6/7 [opendatasets]
   ---------------------------------------- 7/7 [opendatasets]



ModuleNotFoundError: No module named 'cgi'

In [8]:
import opendatasets as od

# This will ask for Kaggle username/key OR download directly
od.download("https://www.kaggle.com/paultimothymooney/chest-xray-pneumonia")

ModuleNotFoundError: No module named 'cgi'